# Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [3]:
library(readxl)
library(ggplot2)
library(viridis)
library(dplyr)
library(glue)
library(tidyr)

# Set Modules and GO Databases

In [4]:
# Subset to significant modules only
significant_associations <- read.table("08_WGCNA/MECP2_significant_associations.txt", header = TRUE, sep = "\t", stringsAsFactors = FALSE)

# Remove ME prefix from module name
significant_associations$module <- sub("^ME", "", significant_associations$module)

# Remove grey module
significant_associations <- significant_associations[significant_associations$module != "grey", ]

# Remove insignificant modules
significant_associations <- significant_associations[significant_associations$pvalue < 0.05, ]

# Create list of significant modules
modules <- unique(significant_associations$module)

# View
print(modules)

 [1] "floralwhite"     "magenta4"        "orange"          "darkgrey"       
 [5] "skyblue3"        "mediumorchid"    "antiquewhite4"   "white"          
 [9] "maroon"          "midnightblue"    "saddlebrown"     "plum2"          
[13] "plum1"           "thistle2"        "lightcyan"       "plum3"          
[17] "navajowhite1"    "indianred4"      "darkorange2"     "darkolivegreen" 
[21] "salmon4"         "lightcyan1"      "skyblue1"        "yellow4"        
[25] "brown4"          "yellowgreen"     "palevioletred3"  "palevioletred2" 
[29] "violet"          "lightsteelblue"  "coral1"          "honeydew1"      
[33] "mediumpurple2"   "skyblue2"        "darkred"         "lightpink3"     
[37] "coral2"          "thistle3"        "darkmagenta"     "darkturquoise"  
[41] "skyblue"         "mediumpurple3"   "royalblue"       "orangered4"     
[45] "paleturquoise"   "bisque4"         "navajowhite2"    "darkgreen"      
[49] "lightsteelblue1" "sienna3"         "salmon"          "tan"            

In [5]:
# List of GO databases
databases <- c("GO_Biological_Process_2023", "GO_Cellular_Component_2023", "GO_Molecular_Function_2023",
               "KEGG_2019_Mouse", "Panther_2016", "RNAseq_DiseaseGene_DrugSigs_GEO")

# Visualize GO Results

In [12]:
# Store GO data into data frame

# Initialize an empty dataframe
all_data <- data.frame()

# Iterate over each module and read in the corresponding Excel file
for (module in modules) {
  file_path <- glue("09_WGCNA_GO/MECP2/{module}_enrichr_results.xlsx")
  
  for (database in databases) {
    try({
      # Read the data from the Excel file
      df <- read_excel(file_path, sheet = database)
      
      if (nrow(df) == 0) {
        next  # Skip to the next database if no data
      }
      
      # Select the required columns and add module and database information
      df <- df %>%
        select(Term, Adjusted.P.value, Odds.Ratio) %>%
        filter(Adjusted.P.value <= 0.05) %>%
        mutate(Module = module, Database = database)
      
      # Append to the dataframe
      all_data <- bind_rows(all_data, df)
      
    }, silent = TRUE)
  }
}

# Check if any data was read
if (nrow(all_data) == 0) {
  stop("No data read from any of the Excel files.")
}

# Display the combined dataframe
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,De novo pyrimidine deoxyribonucleotide biosynthesis Homo sapiens P02739,2.304180e-02,30.598765,floralwhite,Panther_2016
2,Adenine and hypoxanthine salvage pathway Homo sapiens P02723,3.031727e-02,83.062500,magenta4,Panther_2016
3,Fructose galactose metabolism Homo sapiens P02744,4.610824e-02,26.332447,orangered4,Panther_2016
4,Integrin signalling pathway Homo sapiens P00034,3.205042e-02,3.808007,lightyellow,Panther_2016
5,Bmal1 U2OS knockdown GSE56924 down,1.618042e-29,7.414193,pink,RNAseq_DiseaseGene_DrugSigs_GEO
6,Autism iPSCs-derived Neural Cultures Differentiation day 11 GSE61476 up,2.015993e-21,6.017794,pink,RNAseq_DiseaseGene_DrugSigs_GEO


In [13]:
# Count the number of total terms and unique terms for each database
database_term_counts <- all_data %>%
  group_by(Database) %>%
  summarise(
    Total_Terms = n(),
    Unique_Terms = n_distinct(Term)
  )

# Print the table
print(database_term_counts)

# A tibble: 2 × 3
  Database                        Total_Terms Unique_Terms
  <chr>                                 <int>        <int>
1 Panther_2016                             16           15
2 RNAseq_DiseaseGene_DrugSigs_GEO        4917         1183


In [14]:
# Calculate the number of modules each term appears in for each database
term_module_counts <- all_data %>%
  group_by(Database, Term) %>%
  summarise(ModuleCount = n_distinct(Module), .groups = 'drop')

# Rank the terms within each database by the number of modules they appear in
ranked_terms <- term_module_counts %>%
  arrange(Database, desc(ModuleCount)) %>%
  group_by(Database) %>%
  slice_head(n = 50) %>%
  ungroup()

# Merge with the original data to filter the top 25 terms per database
filtered_data_top_25 <- all_data %>%
  semi_join(ranked_terms, by = c("Database", "Term"))

# Print the filtered data
head(filtered_data_top_25)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,De novo pyrimidine deoxyribonucleotide biosynthesis Homo sapiens P02739,0.0230418038,30.598765,floralwhite,Panther_2016
2,Adenine and hypoxanthine salvage pathway Homo sapiens P02723,0.0303172695,83.062500,magenta4,Panther_2016
3,Fructose galactose metabolism Homo sapiens P02744,0.0461082426,26.332447,orangered4,Panther_2016
4,Integrin signalling pathway Homo sapiens P00034,0.0320504231,3.808007,lightyellow,Panther_2016
5,Trim33 B lymphoblastic leukemia cells Knockdown GSE66232 up,0.0001191477,2.677407,pink,RNAseq_DiseaseGene_DrugSigs_GEO
6,TGFB3 Palatal shelves E15.5 GSE45568 down,0.0002724303,2.588162,pink,RNAseq_DiseaseGene_DrugSigs_GEO


In [15]:
# Count the number of total terms and unique terms for each database
filtered_database_term_counts <- filtered_data_top_25 %>%
  group_by(Database) %>%
  summarise(
    Total_Terms = n(),
    Unique_Terms = n_distinct(Term)
  )

# Print the table
print(filtered_database_term_counts)

# A tibble: 2 × 3
  Database                        Total_Terms Unique_Terms
  <chr>                                 <int>        <int>
1 Panther_2016                             16           15
2 RNAseq_DiseaseGene_DrugSigs_GEO         357           50


In [16]:
# Create plots for each database
for (database in unique(filtered_data_top_25$Database)) {
  database_filtered_data <- filtered_data_top_25 %>%
    filter(Database == database)
  
  if (nrow(database_filtered_data) > 0) {
    
    # Calculate dynamic height based on number of unique terms
    n_terms <- length(unique(database_filtered_data$Term))
    plot_height <- ifelse(n_terms > 25, 11, 7)
    
    dot_plot <- ggplot(database_filtered_data, aes(x = Module, y = Term, size = Odds.Ratio, fill = Adjusted.P.value)) +
      geom_point(shape = 21) +
      scale_fill_viridis() +
      xlab('') + ylab('') +
      labs(
        title = 'Top Enrichr Terms Across Modules',
        subtitle = glue('{database} (n_terms = {n_terms})')
      ) +
      theme(
        panel.background = element_rect(fill = "white", color = NA),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        panel.border = element_blank(),
        axis.line = element_line(color = "black"),
        plot.background = element_rect(fill = "white", color = NA),
        axis.text.x = element_text(angle = 90, hjust = 1)
      )
    
    # Save the dot plot with the database name in the filename
    ggsave(filename = glue("09_WGCNA_GO/top_50_dot_plot_{database}.pdf"), 
           plot = dot_plot, 
           height = plot_height, 
           width = 15)
    
    cat(glue("Saved {database} plot with height {plot_height} (n_terms = {n_terms})\n"))
    
  } else {
    cat(glue("No data available for {database}. Skipping...\n"))
  }
}

Saved Panther_2016 plot with height 7 (n_terms = 15)Saved RNAseq_DiseaseGene_DrugSigs_GEO plot with height 11 (n_terms = 50)